# 2 环境准备与模型转换
这一章的目标是准备 ACL 推理所需的两个文件：模型目录中的 `yolov13.om` 和数据目录中的 `bus.bin`。下面的 Cell 有先后关系，后面的路径变量和文件会沿用前面的结果。

## 1. 初始化目录和 CANN 环境
运行前可以在下面的 Cell 中填写 `USER_REPO_ROOT`，或者在启动 Notebook 前设置 `GITCODE_REPO_ROOT`。如果两者都为空，才使用 CANN Lab 的默认挂载路径。Cell 会在选定的案例目录下创建 `workspace`、`model` 和 `data` 目录；CANN 的 `set_env.sh` 会补齐 ATC 环境变量。

In [ ]:
from pathlib import Path
import os, subprocess, sys
python_lib = Path(sys.prefix) / 'lib'
os.environ['LD_LIBRARY_PATH'] = str(python_lib) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
USER_REPO_ROOT = ''  # 可选：例如 '/mnt/workspace/my-repo'
DEFAULT_REPO_ROOT = '/mnt/workspace/gitCode/cann/cann-learning-hub'
REPO_ROOT = Path(USER_REPO_ROOT or os.environ.get('GITCODE_REPO_ROOT', DEFAULT_REPO_ROOT))
os.environ['GITCODE_REPO_ROOT'] = str(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT/'reference_practice'/'yolov13_offline_inference'
WORK_DIR=TUTORIAL_DIR/'workspace'; MODEL_DIR=TUTORIAL_DIR/'model'; DATA_DIR=TUTORIAL_DIR/'data'
for path in (WORK_DIR,MODEL_DIR,DATA_DIR): path.mkdir(parents=True, exist_ok=True)
cann_path = os.environ.get('ASCEND_HOME_PATH')
if not cann_path: raise RuntimeError('ASCEND_HOME_PATH 未设置，请先加载 CANN 的 set_env.sh。')
CANN_ROOT=Path(cann_path)
env_script=CANN_ROOT/'set_env.sh'
if env_script.exists():
    env=subprocess.check_output(['bash','-lc',f'source {env_script} && env'],text=True)
    for line in env.splitlines():
        key,sep,value=line.partition('=')
        if sep: os.environ[key]=value
os.environ['LD_LIBRARY_PATH']=str(python_lib)+':'+os.environ.get('LD_LIBRARY_PATH','')
print(TUTORIAL_DIR)

## 2. 安装模型导出依赖
前一个 Cell 只准备运行环境，还没有安装 YOLOv13 的 Python 包。YOLOv13 使用了自定义网络模块，不能直接用普通 ultralytics 版本加载，因此先拉取官方仓库并以本地源码安装。安装 `thop` 等依赖后，下一步才能导入 `YOLO` 并读取权重。

In [ ]:
YOLOV13_DIR=WORK_DIR/'yolov13'
if not YOLOV13_DIR.exists(): subprocess.run(['git','clone','https://github.com/iMoonLab/yolov13.git',str(YOLOV13_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-deps',str(YOLOV13_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','thop','huggingface_hub','seaborn','py-cpuinfo','opencv-python-headless'],check=True)

## 3. 下载权重并导出 ONNX
现在依赖已经就绪。这个 Cell 先下载 `yolov13n.pt`，再调用 YOLOv13 的导出接口生成固定输入为 `1x3x640x640` 的 ONNX。固定 Shape 与后面的 ATC `--input_shape` 必须一致，否则 ATC 无法按预期编译模型。

In [ ]:
WEIGHT_PATH=WORK_DIR/'yolov13n.pt'; ONNX_PATH=WORK_DIR/'yolov13n.onnx'
if not WEIGHT_PATH.exists(): subprocess.run(['wget','-O',str(WEIGHT_PATH),'https://github.com/iMoonLab/yolov13/releases/download/yolov13/yolov13n.pt'],check=True)
from ultralytics import YOLO
if not ONNX_PATH.exists():
    exported=Path(YOLO(str(WEIGHT_PATH)).export(format='onnx',opset=11,imgsz=640))
    if exported != ONNX_PATH: exported.replace(ONNX_PATH)

## 4. 自动识别 `soc_version`
ATC 的 `--soc_version` 必须对应实际芯片。按照官方建议，先用 `npu-smi info -t board` 读取 `Chip Name` 和 `NPU Name`，再将两者用下划线连接；如果设备不提供板卡信息，则回退到普通设备名称并在前面加上 `Ascend`。多卡环境可以通过 `NPU_ID` 和 `CHIP_ID` 指定查询对象。识别出的值会传给后面的 ATC Cell，不再写死芯片型号。

In [ ]:
import re
npu_env = os.environ.copy()
npu_env['LD_LIBRARY_PATH'] = str(CANN_ROOT/'aarch64-linux'/'lib64') + ':' + npu_env.get('LD_LIBRARY_PATH', '')
npu_id = os.environ.get('NPU_ID')
if not npu_id:
    listed = subprocess.run(['npu-smi', 'info', '-l'], env=npu_env, capture_output=True, text=True, check=True).stdout
    ids = re.findall(r'NPU ID\s*:\s*(\d+)', listed)
    if not ids: raise RuntimeError('无法从 npu-smi info -l 识别 NPU_ID，请设置 NPU_ID。')
    npu_id = ids[0]
chip_id = os.environ.get('CHIP_ID', '0')
board = subprocess.run(['npu-smi', 'info', '-t', 'board', '-i', npu_id, '-c', chip_id], env=npu_env, capture_output=True, text=True, check=True).stdout
chip_name = re.search(r'Chip Name\s*:\s*(\S+)', board)
npu_name = re.search(r'NPU Name\s*:\s*(\S+)', board)
if chip_name and npu_name:
    SOC_VERSION = f'{chip_name.group(1)}_{npu_name.group(1)}'
else:
    info = subprocess.run(['npu-smi', 'info'], env=npu_env, capture_output=True, text=True, check=True).stdout
    name = re.search(r'\|\s*\d+\s+(\S+)', info)
    if not name:
        raise RuntimeError('无法从 npu-smi 输出识别 soc_version，请设置 SOC_VERSION 手动覆盖。')
    SOC_VERSION = name.group(1) if name.group(1).startswith('Ascend') else 'Ascend' + name.group(1)
print(f'NPU_ID={npu_id}, CHIP_ID={chip_id}, SOC_VERSION={SOC_VERSION}')

## 5. 用 ATC 编译离线模型
ONNX 只描述网络结构，ACL 运行时需要加载面向具体昇腾芯片编译出的 OM。下面的命令使用上一节自动识别的 `SOC_VERSION`，`--framework=5` 表示输入模型来自 ONNX，`--output` 是不带扩展名的输出前缀。`--input_shape` 指定输入名和静态 Shape，`--output_type=FP32` 指定输出数据类型。更多参数可参考 [ATC 参数说明](https://www.hiascend.com/document/detail/zh/canncommercial/latest/devaids/atctool/atlasatcparam_16_0036.html)。Cell 成功后，`model/yolov13.om` 才能被 ACL 加载。

In [ ]:
OM_PREFIX=MODEL_DIR/'yolov13'
subprocess.run(['atc','--model='+str(ONNX_PATH),'--framework=5','--output='+str(OM_PREFIX),'--soc_version='+SOC_VERSION,'--input_shape=images:1,3,640,640','--output_type=FP32'],check=True)
print(OM_PREFIX.with_suffix('.om'))

## 6. 下载测试图片并生成 ACL 输入
模型已经转换完成，接下来准备一张与模型输入对应的图片。预处理脚本会把图片做 letterbox 缩放、转换为 RGB、归一化并从 HWC 排列转成 NCHW，最后写入 FP32 的 `bus.bin`；同时保存缩放比例和 padding，供后处理映射回原图。脚本必须在 `data` 目录执行，因为它会扫描当前目录中的图片。

处理完成后，Cell 会直接显示原始 `bus.jpg`，方便在推理前确认输入内容。这里显示的图片还没有检测框，第 4 章会展示带识别结果的图片。

In [ ]:
from IPython.display import Image, display
IMAGE_PATH=DATA_DIR/'bus.jpg'
if not IMAGE_PATH.exists(): subprocess.run(['wget','-O',str(IMAGE_PATH),'https://raw.githubusercontent.com/iMoonLab/yolov13/main/ultralytics/assets/bus.jpg'],check=True)
subprocess.run([sys.executable,str(TUTORIAL_DIR/'scripts'/'preprocess.py')],cwd=DATA_DIR,check=True)
print(IMAGE_PATH.with_suffix('.bin'))
display(Image(filename=str(IMAGE_PATH), width=640))